In [8]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.metrics import accuracy_score
import joblib

print("Deploying Regularized 'Goldilocks' Model for 60%+ Target...")

# 1. Load Data
matches = pd.read_csv('../data/raw/matches.csv')
venue_intel = pd.read_csv('../data/processed/venue_intelligence.csv')
form_df = pd.read_csv('../data/processed/team_form.csv')
dominance_df = pd.read_csv('../data/processed/dominance_matrix.csv')

# Standardize teams
team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals', 'Kings XI Punjab': 'Punjab Kings', 
    'Deccan Chargers': 'Sunrisers Hyderabad', 'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru', 'Gujarat Lions': 'Gujarat Titans'
}
matches.replace(team_mapping, inplace=True)
matches.dropna(subset=['winner', 'venue', 'date'], inplace=True)

# 2. THE GOLDILOCKS ERA FILTER (2015 Onwards)
matches['date'] = pd.to_datetime(matches['date'])
matches['year'] = matches['date'].dt.year
matches = matches[matches['year'] >= 2015].copy() # Enough data to learn, but modern enough to be relevant

# 3. FEATURE ENGINEERING: Home Advantage & Toss
home_venues = {
    'Chennai Super Kings': 'MA Chidambaram Stadium', 'Mumbai Indians': 'Wankhede Stadium',
    'Royal Challengers Bengaluru': 'M Chinnaswamy Stadium', 'Kolkata Knight Riders': 'Eden Gardens',
    'Delhi Capitals': 'Arun Jaitley Stadium', 'Rajasthan Royals': 'Sawai Mansingh Stadium',
    'Punjab Kings': 'Punjab Cricket Association Stadium, Mohali', 'Sunrisers Hyderabad': 'Rajiv Gandhi International Stadium',
    'Gujarat Titans': 'Narendra Modi Stadium', 'Lucknow Super Giants': 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium'
}
matches['team1_home'] = matches.apply(lambda x: 1 if (home_venues.get(x['team1']) and home_venues.get(x['team1']) in str(x['venue'])) else 0, axis=1)
matches['team2_home'] = matches.apply(lambda x: 1 if (home_venues.get(x['team2']) and home_venues.get(x['team2']) in str(x['venue'])) else 0, axis=1)
matches['team1_won_toss'] = (matches['toss_winner'] == matches['team1']).astype(int)

# 4. Inject Venue DNA & Form
venue_win_dict = dict(zip(venue_intel['venue'], venue_intel['bat_first_win_pct']))
matches['venue_bat_first_win_pct'] = matches['venue'].map(venue_win_dict).fillna(50.0)

matches = pd.merge(matches, form_df[['id', 'team', 'rolling_5_form']], left_on=['id', 'team1'], right_on=['id', 'team'], how='left').drop('team', axis=1).rename(columns={'rolling_5_form': 'team1_form'})
matches = pd.merge(matches, form_df[['id', 'team', 'rolling_5_form']], left_on=['id', 'team2'], right_on=['id', 'team'], how='left').drop('team', axis=1).rename(columns={'rolling_5_form': 'team2_form'})
matches['team1_form'].fillna(0.5, inplace=True)
matches['team2_form'].fillna(0.5, inplace=True)
matches['form_diff'] = matches['team1_form'] - matches['team2_form']

matches['matchup'] = matches.apply(lambda x: ' vs '.join(sorted([str(x['team1']), str(x['team2'])])), axis=1)
dom_team1 = dominance_df.rename(columns={'winner': 'team', 'dominance_score': 'team1_dominance'})
matches = pd.merge(matches, dom_team1[['matchup', 'team', 'team1_dominance']], left_on=['matchup', 'team1'], right_on=['matchup', 'team'], how='left').drop('team', axis=1)
matches['team1_dominance'].fillna(0.5, inplace=True)
matches.drop_duplicates(subset=['id'], inplace=True)

# 5. Target & Weights
matches['target'] = (matches['team1'] == matches['winner']).astype(int)
max_year = matches['year'].max()
matches['sample_weight'] = np.where(max_year - matches['year'] <= 2, 2.0, 1.0) 

# 6. Final Super-Features Array
ml_df = matches[['team1', 'team2', 'venue', 'toss_decision', 'venue_bat_first_win_pct', 
                 'team1_home', 'team2_home', 'team1_won_toss', 
                 'form_diff', 'team1_dominance', 'target', 'sample_weight']].copy()

# 7. Split Data
X = ml_df.drop('target', axis=1)
y = ml_df['target']

split_idx = int(len(ml_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

weights_train = X_train['sample_weight'].values
X_train = X_train.drop('sample_weight', axis=1)
X_test = X_test.drop('sample_weight', axis=1)

# 8. Preprocessing
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), ['team1', 'team2', 'venue', 'toss_decision']),
    ('num', StandardScaler(), ['venue_bat_first_win_pct', 'form_diff', 'team1_dominance', 'team1_home', 'team2_home', 'team1_won_toss'])
], remainder='passthrough')

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# 9. HIGHLY REGULARIZED MODELS (To prevent the accuracy crash)
# Notice the strict limits on depth, the C value for LR, and reg_lambda for XGBoost
model_lr = LogisticRegression(max_iter=2000, C=0.1, random_state=42)
model_rf = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_split=10, random_state=42)
model_xgb = xgb.XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, reg_lambda=5.0, subsample=0.8, eval_metric='logloss', random_state=42)

ensemble = VotingClassifier(
    estimators=[('lr', model_lr), ('rf', model_rf), ('xgb', model_xgb)],
    voting='soft'
)

# 10. Train and Evaluate
print("Training Balanced Master Ensemble...")
ensemble.fit(X_train_transformed, y_train, sample_weight=weights_train)

y_pred = ensemble.predict(X_test_transformed)
accuracy = accuracy_score(y_test, y_pred)
print(f"🚀 BALANCED Ensemble Accuracy: {accuracy * 100:.2f}%")

# Save
joblib.dump(preprocessor, '../models/preprocessor.pkl')
joblib.dump(ensemble, '../models/weighted_ensemble.pkl')
joblib.dump(list(X_train.columns), '../models/ensemble_features.pkl')
print("✅ Optimized model saved successfully!")

Deploying Regularized 'Goldilocks' Model for 60%+ Target...
Training Balanced Master Ensemble...


C:\Users\garvi\AppData\Local\Temp\ipykernel_20620\379124954.py:53: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  matches['team1_form'].fillna(0.5, inplace=True)
C:\Users\garvi\AppData\Local\Temp\ipykernel_20620\379124954.py:54: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exam

🚀 BALANCED Ensemble Accuracy: 58.27%
✅ Optimized model saved successfully!
